# 🔦 Parte 3 — O mecanismo de atenção: Query, Key e Value

Chegou a hora do coração do Transformer: a **atenção** (*self-attention*).

Até agora sabemos:
- transformar palavras em vetores (**Parte 1**);
- calcular **pesos de atenção** com produto escalar + softmax (**Parte 2**).

Agora vamos usar esses pesos para **criar um novo vetor para cada palavra**, que carrega o **contexto da frase**.

📌 Execute as células na ordem com `Shift + Enter`.

In [ ]:
import math

#                [animal, comida, ação]
embeddings = {
    "o":         [0.1,    0.1,    0.1],
    "gato":      [0.9,    0.1,    0.0],
    "cachorro":  [0.8,    0.1,    0.1],
    "leite":     [0.0,    0.9,    0.1],
    "queijo":    [0.1,    0.8,    0.0],
    "bebeu":     [0.1,    0.3,    0.9],
    "comeu":     [0.1,    0.4,    0.8],
}

def produto_escalar(vetor_a, vetor_b):
    soma = 0
    for i in range(len(vetor_a)):
        soma = soma + vetor_a[i] * vetor_b[i]
    return round(soma, 2)

def softmax(lista_de_escores):
    exponenciais = []
    for valor in lista_de_escores:
        exponenciais.append(math.exp(valor))
    total = 0
    for valor in exponenciais:
        total = total + valor
    pesos = []
    for valor in exponenciais:
        pesos.append(round(valor / total, 3))
    return pesos

def mostrar_barras(palavras, pesos):
    for i in range(len(palavras)):
        porcentagem = pesos[i] * 100
        barra = "█" * int(porcentagem / 2)
        print("   ", palavras[i].ljust(10), barra, round(porcentagem, 1), "%")

print("Funções das Partes 1 e 2 carregadas!")

## Passo 1 — A soma ponderada: misturando as palavras

Os pesos da Parte 2 dizem **quanto** cada palavra olha para as outras. Agora vamos **usar** esses pesos:

> O novo vetor de uma palavra = (peso₁ × vetor₁) + (peso₂ × vetor₂) + (peso₃ × vetor₃) ...

Isso se chama **soma ponderada**: cada palavra "contribui" com uma parte proporcional à atenção que recebeu.

Vamos fazer para a palavra **"bebeu"** na frase **"gato bebeu leite"**, passo a passo:

In [ ]:
frase = ["gato", "bebeu", "leite"]
foco = "bebeu"

# 1) escores e pesos (igual à Parte 2)
escores = []
for palavra in frase:
    escores.append(produto_escalar(embeddings[foco], embeddings[palavra]))
pesos = softmax(escores)
print("Pesos de atenção de '" + foco + "':")
mostrar_barras(frase, pesos)

# 2) soma ponderada
novo_vetor = [0, 0, 0]
for i in range(len(frase)):
    vetor = embeddings[frase[i]]
    for posicao in range(3):
        novo_vetor[posicao] = novo_vetor[posicao] + pesos[i] * vetor[posicao]

for posicao in range(3):
    novo_vetor[posicao] = round(novo_vetor[posicao], 2)

print()
print("Vetor ANTES da atenção :", embeddings[foco])
print("Vetor DEPOIS da atenção:", novo_vetor)

🔎 O vetor de "bebeu" **mudou**: agora ele tem um pouco de "animal" (veio do *gato*) e mais "comida" (veio do *leite*).
Ou seja, o novo vetor **sabe quem bebeu e o que foi bebido**. Isso é **contexto**!

## Passo 2 — Por que isso é tão importante? O exemplo do "banco" 🏦🪑

A palavra **"banco"** tem dois significados:
- *"Saquei dinheiro no **banco**"* → instituição financeira;
- *"Sentei no **banco** da praça"* → assento.

O embedding de "banco" é **sempre o mesmo**, mas depois da atenção ele vai **mudar conforme a frase**. Vamos testar!

Usaremos 3 características: `[dinheiro, descanso, lugar ao ar livre]`.

In [ ]:
#                         [dinheiro, descanso, ar livre]
embeddings_banco = {
    "banco":    [1.0, 1.0, 0.5],    # ambíguo: um pouco de tudo
    "saquei":   [2.0, 0.0, 0.0],
    "dinheiro": [3.0, 0.0, 0.0],
    "sentei":   [0.0, 2.0, 0.0],
    "praça":    [0.0, 1.0, 2.5],
    "no":       [0.1, 0.1, 0.1],
    "da":       [0.1, 0.1, 0.1],
}

def atencao_simples(frase, foco, tabela_embeddings):
    escores = []
    for palavra in frase:
        escores.append(produto_escalar(tabela_embeddings[foco], tabela_embeddings[palavra]))
    pesos = softmax(escores)
    print("Pesos de atenção de '" + foco + "':")
    mostrar_barras(frase, pesos)

    novo_vetor = [0, 0, 0]
    for i in range(len(frase)):
        vetor = tabela_embeddings[frase[i]]
        for posicao in range(3):
            novo_vetor[posicao] = novo_vetor[posicao] + pesos[i] * vetor[posicao]
    for posicao in range(3):
        novo_vetor[posicao] = round(novo_vetor[posicao], 2)
    return novo_vetor

frase_a = ["saquei", "dinheiro", "no", "banco"]
frase_b = ["sentei", "no", "banco", "da", "praça"]

print("Frase A:", " ".join(frase_a))
vetor_a = atencao_simples(frase_a, "banco", embeddings_banco)
print()
print("Frase B:", " ".join(frase_b))
vetor_b = atencao_simples(frase_b, "banco", embeddings_banco)

print()
print("                     [dinheiro, descanso, ar livre]")
print("'banco' original   :", embeddings_banco["banco"])
print("'banco' na frase A :", vetor_a)
print("'banco' na frase B :", vetor_b)

🎉 **Mesma palavra, vetores diferentes!**
- Na frase A, o "banco" ficou com muito **dinheiro**;
- Na frase B, o "banco" ficou com mais **descanso** e **ar livre**.

É exatamente isso que permite ao ChatGPT, ao Google Tradutor etc. entenderem palavras com vários significados.

---
## Passo 3 — Query, Key e Value (Q, K, V) 📚

A atenção simples tem um problema que já vimos na Parte 2: **cada palavra olha principalmente para si mesma**. Além disso, ela usa o **mesmo vetor** para três papéis diferentes.

O Transformer de verdade cria **três versões** de cada palavra. Pense em uma **biblioteca**:

| Papel | Na biblioteca | Pergunta que responde |
|---|---|---|
| **Query (Q)** — consulta | O que você digita na busca | *"O que eu estou procurando?"* |
| **Key (K)** — chave | A etiqueta na lombada do livro | *"O que eu tenho para oferecer?"* |
| **Value (V)** — valor | O conteúdo do livro | *"Que informação eu entrego?"* |

- Os **escores** passam a ser: `Query da palavra em foco` · `Key de cada palavra`.
- A **soma ponderada** passa a usar os **Values**.

Cada versão é criada multiplicando o embedding por uma **matriz de pesos** (W_Q, W_K, W_V).

### Como multiplicar uma matriz por um vetor?

Uma matriz é uma **lista de linhas**. Para multiplicar por um vetor, fazemos o **produto escalar de cada linha com o vetor**. Simples assim!

In [ ]:
def multiplicar_matriz_vetor(matriz, vetor):
    resultado = []
    for linha in matriz:
        resultado.append(produto_escalar(linha, vetor))
    return resultado

# Teste rápido
matriz_teste = [[1, 0, 0],
                [0, 2, 0],
                [0, 0, 3]]
print(multiplicar_matriz_vetor(matriz_teste, [1, 1, 1]))   # esperado: [1, 2, 3]

### As nossas matrizes W_Q, W_K e W_V

Em um modelo real, essas matrizes são **aprendidas no treinamento**. Aqui vamos montar a **W_Q à mão** com uma ideia linguística simples:

- Um **animal** ou uma **comida** procura uma **ação** (*"o que fizeram comigo?"*);
- Uma **ação** procura **animais e comidas** (*"quem fez? com o quê?"*).

Para W_K e W_V usamos a **matriz identidade** (só tem 1 na diagonal), que **não altera** o vetor. Assim fica mais fácil enxergar o efeito da Query.

In [ ]:
# Cada LINHA gera uma posição da Query: [procura animal, procura comida, procura ação]
W_Q = [
    [0, 0, 3],   # "procura animal"  <- vem de ser uma AÇÃO
    [0, 0, 3],   # "procura comida"  <- vem de ser uma AÇÃO
    [3, 3, 0],   # "procura ação"    <- vem de ser ANIMAL ou COMIDA
]

W_K = [[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]]

W_V = [[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]]

frase = ["gato", "bebeu", "leite"]

Q = {}
K = {}
V = {}
for palavra in frase:
    Q[palavra] = multiplicar_matriz_vetor(W_Q, embeddings[palavra])
    K[palavra] = multiplicar_matriz_vetor(W_K, embeddings[palavra])
    V[palavra] = multiplicar_matriz_vetor(W_V, embeddings[palavra])
    print(palavra.ljust(6), "| Q =", Q[palavra], "| K =", K[palavra], "| V =", V[palavra])

## Passo 4 — A divisão pela raiz (escala)

Na Parte 2 vimos que escores grandes deixam o softmax "decidido demais". Nos modelos reais, os vetores têm centenas de posições, e o produto escalar fica **enorme**.

Por isso o Transformer **divide os escores por √d** (raiz quadrada do tamanho do vetor). No nosso caso, d = 3, então dividimos por √3 ≈ 1.73.

A fórmula completa da atenção, do artigo *"Attention Is All You Need"* (2017), é:

$$\text{Atenção}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K}{\sqrt{d}}\right) \cdot V$$

Parece assustadora, mas **você já sabe fazer cada pedaço**! Vamos calcular para "bebeu":

In [ ]:
foco = "bebeu"
d = 3

# 1) escores: Query do foco x Key de cada palavra, divididos por raiz de d
escores = []
for palavra in frase:
    escore = produto_escalar(Q[foco], K[palavra]) / math.sqrt(d)
    escores.append(round(escore, 2))
    print("Q[" + foco + "] · K[" + palavra + "] / √3 =", round(escore, 2))

# 2) softmax
pesos = softmax(escores)
print()
print("Pesos de atenção de '" + foco + "':")
mostrar_barras(frase, pesos)

# 3) soma ponderada dos VALUES
saida = [0, 0, 0]
for i in range(len(frase)):
    for posicao in range(3):
        saida[posicao] = saida[posicao] + pesos[i] * V[frase[i]][posicao]
for posicao in range(3):
    saida[posicao] = round(saida[posicao], 2)

print()
print("Saída da atenção para '" + foco + "':", saida)

✅ Agora **"bebeu" presta mais atenção em "gato" e "leite"** do que nela mesma! A Query fez a palavra "procurar" o que realmente importa.

## Passo 5 — Atenção para a frase inteira

Vamos juntar tudo em uma função e calcular a atenção para **todas** as palavras:

In [ ]:
def atencao(frase, tabela_embeddings, W_Q, W_K, W_V, mostrar=True):
    d = len(tabela_embeddings[frase[0]])

    # Cria Q, K e V de cada palavra
    Q = {}
    K = {}
    V = {}
    for palavra in frase:
        Q[palavra] = multiplicar_matriz_vetor(W_Q, tabela_embeddings[palavra])
        K[palavra] = multiplicar_matriz_vetor(W_K, tabela_embeddings[palavra])
        V[palavra] = multiplicar_matriz_vetor(W_V, tabela_embeddings[palavra])

    saidas = {}
    for foco in frase:
        escores = []
        for palavra in frase:
            escores.append(round(produto_escalar(Q[foco], K[palavra]) / math.sqrt(d), 2))
        pesos = softmax(escores)

        saida = []
        for posicao in range(d):
            soma = 0
            for i in range(len(frase)):
                soma = soma + pesos[i] * V[frase[i]][posicao]
            saida.append(round(soma, 2))
        saidas[foco] = saida

        if mostrar:
            print("Atenção de '" + foco + "':")
            mostrar_barras(frase, pesos)
            print("    saída:", saida)
            print()
    return saidas

resultado = atencao(["gato", "bebeu", "leite"], embeddings, W_Q, W_K, W_V)

🔎 Interpretação:
- **gato** olha principalmente para **bebeu** → "o que o gato fez?";
- **bebeu** olha para **gato** e **leite** → "quem bebeu? o quê?";
- **leite** olha principalmente para **bebeu** → "o que fizeram com o leite?".

Cada palavra saiu da atenção **"sabendo" do resto da frase**.

---
## ✏️ Exercício

1. Rode a função `atencao` com a frase `["cachorro", "comeu", "queijo"]`. Os resultados fazem sentido?
2. Troque o número `3` da `W_Q` por `1` e depois por `10`. O que acontece com as barras? Por quê?
3. Desafio: invente uma `W_V` diferente da identidade (ex.: que dobre a característica "ação") e observe como as saídas mudam.

In [ ]:
# Complete aqui o exercício

---
## 📝 Resumo da Parte 3

| Conceito | O que é |
|---|---|
| **Soma ponderada** | Mistura os vetores das palavras de acordo com os pesos de atenção |
| **Query (Q)** | O que a palavra está procurando |
| **Key (K)** | O que a palavra oferece para ser encontrada |
| **Value (V)** | A informação que a palavra entrega |
| **W_Q, W_K, W_V** | Matrizes que criam Q, K e V (aprendidas no treinamento) |
| **÷ √d** | Escala que impede escores grandes demais |

**A receita da atenção:** `escores = Q·K / √d` → `pesos = softmax(escores)` → `saída = soma ponderada dos V`

➡️ **Na Parte 4** vamos montar um **bloco Transformer completo**: posição das palavras, várias "cabeças" de atenção e a previsão da próxima palavra.